# Homework: Evaluation

## Setup

In [1]:
import json
from tqdm.auto import tqdm
from evaluation.evaluation import generate_ground_truth, calc_total_price
import openai
import os
from models.question import Questions
import pandas as pd

In [27]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
len(documents)

72

In [28]:
documents[:3]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [29]:
doc = documents[0]
user_prompt = json.dumps(doc)
type(user_prompt)

str

### Generating ground truth

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

## Q1. Generating questions

In [ ]:
docs = []
for i in range(3):
    doc = {
        "filename": documents[i]["filename"],
        "content": documents[i]["content"],
    }
    docs.append(doc)

print(docs)

In [ ]:
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
model = os.environ["OPENAI_LLM_MODEL"]

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(docs):
    records, usage = generate_ground_truth(
        doc, client, data_gen_instructions, output_type=Questions, model=model
    )
    ground_truth.extend(records)
    usages.append(usage)

```bash
[ResponseUsage(input_tokens=1023, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=106, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1129),
 ResponseUsage(input_tokens=1289, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=76, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1365),
 ResponseUsage(input_tokens=1756, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1865)]

```


In [ ]:
avg_tokens = sum([usage.total_tokens for usage in usages]) / len(usages)
avg_tokens

In [ ]:
usages

In [ ]:
with (
    open("./results/ground_truth.json", "w", encoding="utf-8") as f1,
    open("./results/usage.json", "w", encoding="utf-8") as f2,
):
    json.dump(ground_truth, f1, indent=4)
    # json.dump(usages, f2, indent=4)

In [ ]:
total_cost = calc_total_price(
    usages, input_price_per_million=0.4, output_price_per_million=0.60
)
total_cost

## Q2. First result with text search

In [80]:
from embeddings.embedding import Embedder

model_path = "..\\.models\\Xenova\\all-MiniLM-L6-v2"

embed = Embedder(model_path)

### Generating embeddings

In [81]:
df_ground_truth = pd.read_csv(".\\results\ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[:3]

<>:1: SyntaxWarning: invalid escape sequence '\g'
<>:1: SyntaxWarning: invalid escape sequence '\g'
C:\Users\Admin\AppData\Local\Temp\ipykernel_20272\2265135234.py:1: SyntaxWarning: invalid escape sequence '\g'
  df_ground_truth = pd.read_csv(".\\results\ground-truth.csv")


[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'}]

In [82]:
import requests
from minsearch import Index, VectorSearch

def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [83]:
def text_search(query, num_results=5):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict
    )

In [84]:
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    results = vindex.search(query_vector, num_results=num_results)

    return results
    

In [85]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

texts = [chunk["content"] for chunk in chunks]
print("Number of chunks:", len(texts))

Number of chunks: 295


In [86]:
X = embed.encode_batch(texts)
X.shape

(295, 384)

In [87]:
index = build_index(documents)
index.fit(chunks)

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

In [117]:
# modify this function to match result_lists
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start"))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [118]:
def hybrid_search(query, k=60, num_results=10):
    text_results = text_search(query, num_results=5)
    vector_results = vector_search(query, num_results=5)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

In [90]:
q = ground_truth[0]["question"]
results_text = text_search(query=q)
results_text

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [91]:
q = ground_truth[0]["question"]
results_vector = vector_search(query=q)
results_vector

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

## Q4. Evaluating text search

In [92]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [93]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[0, 0, 0, 0, 1]

In [94]:
from tqdm.auto import tqdm


def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [95]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [96]:
relevance_total_text

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [97]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [98]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [99]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 0, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [1, 1, 0, 0, 1],
 [1, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 1, 1, 0, 0],
 [1, 1, 0, 0, 0],
 [0, 1, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 1, 1, 0, 0],
 [0, 0, 0, 0, 0]]

In [100]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [101]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [102]:
hit_rate(relevance_total)

0.7583333333333333

In [103]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [104]:
mrr(relevance_total)
# 0.822

0.5942592592592594

In [105]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [106]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [108]:
q = ground_truth[0]["question"]
results = hybrid_search(q)
results

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [119]:
run = {}
for k in [1, 50, 60, 100, 200]:
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q["filename"]
        results = hybrid_search(query=q["question"], k=k)

        relevance = []
        for d in results:
            relevance.append(int(d["filename"] == doc_id))

        relevance_total.append(relevance)

    score = mrr(relevance_total)
    run[k] = score

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [120]:
run

{1: 0.6529453262786599,
 50: 0.6538712522045855,
 60: 0.6538712522045855,
 100: 0.6538712522045855,
 200: 0.6538712522045855}

In [116]:
mrr(relevance_total)

0.6467592592592594